# Relevance Training on Google Colab
Use a free Colab T4 to train the relevance model with Drive-backed checkpoints.


## Flow
- mount Drive
- clone repo
- install relevance training dependencies
- optionally build `v11`, `v9`, or `v10` datasets
- generate a Colab config
- train relevance
- zip/download the final checkpoint


In [23]:
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/fact_checking_system_colab'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import os
import shutil

REPO_URL = 'https://github.com/injetiharsha/fact_checking_system.git'
BRANCH = 'feat/reduce-heuristics-phased'
REPO_DIR = '/content/fact_checking_system'

os.chdir('/content')
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())


Cloning into '/content/fact_checking_system'...
remote: Enumerating objects: 851, done.
remote: Counting objects: 100% (851/851), done.
remote: Compressing objects: 100% (470/470), done.
remote: Total 851 (delta 445), reused 756 (delta 352), pack-reused 0 (from 0)
Receiving objects: 100% (851/851), 815.83 KiB | 13.16 MiB/s, done.
Resolving deltas: 100% (445/445), done.
cwd: /content/fact_checking_system


In [25]:
import os, torch
os.environ['PYTHONWARNINGS'] = 'ignore'
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


CUDA available: True
CUDA device: Tesla T4


In [26]:
!pip install -q --upgrade pip
!pip uninstall -y peft bitsandbytes sentence-transformers > /dev/null 2>&1 || true
!pip install -q transformers==4.38.2 datasets==2.17.1 accelerate==0.27.2 scikit-learn==1.4.2 sentencepiece==0.2.0 PyYAML==6.0.2 tqdm==4.66.2


In [27]:
# Optional: rebuild the cleaned Phase 2 dataset locally in Colab
!python training/common/build_relevance_v11_phase2.py


Wrote 129 train, 27 validation, and 27 test records to data/relevance/v11


In [28]:
# If you have AVeriTeC files available in the repo, rebuild v10 too.
# !python training/common/build_relevance_v10_averitec.py --averitec-file data/public/averitec/train.json --averitec-file data/public/averitec/dev.json


In [29]:
BASE_CONFIG = 'training/configs/relevance_v11.yaml'
# Example for earlier smaller run:
# BASE_CONFIG = 'training/configs/relevance_v9.yaml'
# Example for public-data run:
# BASE_CONFIG = 'training/configs/relevance_v10_averitec.yaml'
!python training/common/generate_colab_relevance_config.py --base-config {BASE_CONFIG} --drive-dir {DRIVE_DIR}


seed: 42
model:
  name: microsoft/deberta-v3-small
  fallback_models:
  - distilbert-base-uncased
data:
  train_file: data/relevance/v11/train.jsonl
  validation_file: data/relevance/v11/validation.jsonl
  test_file: data/relevance/v11/test.jsonl
training:
  batch_size: 16
  eval_batch_size: 16
  learning_rate: 2.0e-05
  epochs: 50
  max_length: 256
  logging_steps: 20
  early_stopping_patience: 5
  save_total_limit: 2
output:
  checkpoint_dir: /content/drive/MyDrive/fact_checking_system_colab/checkpoints/relevance/v11_run1
  metrics_dir: /content/drive/MyDrive/fact_checking_system_colab/training_artifacts/relevance/v11_run1



In [30]:
COLAB_CONFIG = BASE_CONFIG.replace('.yaml', '_colab.yaml')
!python training/relevance/train.py --config {COLAB_CONFIG}


2026-03-21 17:09:51.961617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774112991.982873   17286 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774112991.989915   17286 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774112992.009954   17286 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774112992.009979   17286 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774112992.009983   17286 computation_placer.cc:177] computation placer alr

In [31]:
import os
import shutil
from pathlib import Path
import yaml

with open(COLAB_CONFIG, 'r', encoding='utf-8') as handle:
    cfg = yaml.safe_load(handle)

checkpoint_dir = Path(cfg['output']['checkpoint_dir'])
run_name = checkpoint_dir.name
archive_path = f'/content/{run_name}.zip'
if checkpoint_dir.exists():
    shutil.make_archive(f'/content/{run_name}', 'zip', checkpoint_dir)
    print('Created in Colab runtime:', archive_path)
    print('Checkpoint source:', checkpoint_dir)
    print('Browser download target after files.download(...): your local Downloads folder')
    get_ipython().system(f'ls -lh {archive_path}')
else:
    print('Checkpoint not found:', checkpoint_dir)


Created in Colab runtime: /content/v11_run1.zip
Checkpoint source: /content/drive/MyDrive/fact_checking_system_colab/checkpoints/relevance/v11_run1
Browser download target after files.download(...): your local Downloads folder
-rw-r--r-- 1 root root 1.7G Mar 21 17:20 /content/v11_run1.zip


In [32]:
from google.colab import files
from pathlib import Path

archive_base = f"/content/{checkpoint_dir.name}"
archive_path = archive_base + ".zip"

print('Archive inside Colab runtime:', archive_path)
print('This downloads through the browser to your normal Downloads folder.')
if Path(archive_path).exists():
    files.download(archive_path)
else:
    print('Archive not found. Run the previous archive-creation cell first.')


Archive inside Colab runtime: /content/v11_run1.zip
This downloads through the browser to your normal Downloads folder.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
drive_zip = Path("/content/drive/MyDrive/fact_checking_system_colab/v11_run1_min.zip")
shutil.copy2("/content/v11_run1.zip", drive_zip)
print("Saved to Drive:", drive_zip)


Saved to Drive: /content/drive/MyDrive/fact_checking_system_colab/v11_run1_min.zip
